# Rating Plausibility of Word Senses in Ambiguous Sentences through Narrative Understanding
## SemEval 2026 -- Task 05

The task at hand is to evaluate the plausibility of a given meaning for a homonym within a narrative context. Specifically, it involves:

Goal: To predict a plausibility rating (on a scale of 1 to 5) for a homonym's meaning within a given sentence and its surrounding context.
Approach: Using a DebertaV3-base model, fine-tuned for sequence classification, with different prompting strategies to format the input text.

In [1]:
!pip install -q transformers[torch] datasets sentencepiece pandas scipy


!git clone https://github.com/Janosch-Gehring/semeval26-05-scripts.git

Cloning into 'semeval26-05-scripts'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 51 (delta 7), reused 4 (delta 4), pack-reused 36 (from 1)
Receiving objects: 100% (51/51), 268.49 KiB | 4.79 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
%cd semeval26-05-scripts

/content/semeval26-05-scripts


In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
import json
import os

import pandas as pd
import numpy as np

import torch
from datasets import Dataset

from google.colab import files
import shutil

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

### Prompt Types:

1.  **Simple Prompt (Approach 1)**: This approach concatenates the `precontext`, `sentence`, `ending`, and `judged_meaning` fields directly, separated by `[SEP]`, to form the input text. The structure is `context sentence ending [SEP] meaning`. This is a straightforward concatenation of the available text segments.



In [ ]:
# approach 1: simple prompt
def load_ambistory_from_local_dict(data_dir="data"):
    dataset = {}

    for split in ['train', 'dev', 'test']:
        file_path = os.path.join(data_dir, f"{split}.json")

        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue

        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        rows = []
        for key, entry in raw_data.items():

            context = entry.get('precontext', "")
            target_sent = entry.get('sentence', "")
            meaning = entry.get('judged_meaning', "")
            ending = entry.get('ending', "")
            rating = entry.get('average', 0.0)
            # simple prompt structure
            rich_input = f"{context} {target_sent} {ending} [SEP] {meaning}"

            if split in ['train', 'dev']:
              rows.append({
                  "text_input": rich_input,
                  "label": float(entry.get('average', 0.0)),
                  "homonym": entry.get('homonym', "")
              })
            else:
              rows.append({
                  "id": key, # Add the key as 'id' for test instances
                  "text_input": rich_input,
                  "homonym": entry.get('homonym', "")
              })

        dataset[split] = pd.DataFrame(rows)
        print(f"Successfully loaded {split}: {len(dataset[split])} rows.")

    return dataset

data_dict = load_ambistory_from_local_dict("data")
train_df = data_dict.get('train')
dev_df = data_dict.get('dev')
test_df = data_dict.get('test')

if train_df is not None:
    print("\nSample processed input:")
    print(train_df['text_input'].iloc[0])
    print(f"Label: {train_df['label'].iloc[0]}")

Successfully loaded train: 2280 rows.
Successfully loaded dev: 588 rows.
Successfully loaded test: 930 rows.

Sample processed input:
The old machine hummed in the corner of the workshop. Clara examined its dusty dials with a furrowed brow. She wondered if it could be brought back to life. The potential couldn't be measured. She collected a battery reader and looked on earnestly, willing some life back into the old machine. [SEP] the difference in electrical charge between two points in a circuit expressed in volts
Label: 3.0


2.  **Instructional Prompt (Approach 2)**: This method constructs a more natural language prompt, explicitly framing the task as a question. It includes the narrative elements (`precontext`, `sentence`, `ending`) and then asks, "On a scale of 1 to 5, how plausible is the following meaning for the word '[homonym]'? Meaning: [judged_meaning]". This guides the model with a clear instruction.


In [ ]:
# approach 2: instructional prompt
def load_ambistory_instructional(data_dir="data"):
    dataset = {}
    for split in ['train', 'dev', 'test']:
        file_path = os.path.join(data_dir, f"{split}.json")
        if not os.path.exists(file_path): continue

        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        rows = []
        for key, entry in raw_data.items():

            prompt = (
                f"Narrative: {entry.get('precontext', '')} {entry.get('sentence', '')} {entry.get('ending', '')} "
                f"Question: On a scale of 1 to 5, how plausible is the following meaning for the word '{entry.get('homonym', '')}'? "
                f"Meaning: {entry.get('judged_meaning', '')}"
            )

            if split in ['train', 'dev']:
              rows.append({
                  "text_input": prompt,
                  "label": float(entry.get('average', 0.0))
              })
            else:
              rows.append({
                  "text_input": prompt
              })
        dataset[split] = pd.DataFrame(rows)
        print(f"Successfully loaded {split}: {len(dataset[split])} rows.")

    return dataset

data_dict = load_ambistory_instructional("data")
train_df, dev_df, test_df = data_dict['train'], data_dict['dev'], data_dict['test']

Successfully loaded train: 2280 rows.
Successfully loaded dev: 588 rows.
Successfully loaded test: 930 rows.



3.  **Masked Prompt (Approach 3)**: This approach modifies the instructional prompt by replacing the target homonym in the `sentence` with the `[MASK]` token. The prompt structure is similar to the instructional one, but the question changes to "How plausible is this meaning for the missing word?" This technique forces the model to focus on the context surrounding the masked word to evaluate the meaning's plausibility.

In [11]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# approach 3: masked prompt
def load_masked_ambistory(data_dir="data"):
    dataset = {}
    for split in ['train', 'dev', 'test']:
        file_path = f"{data_dir}/{split}.json"
        with open(file_path, 'r') as f:
            raw_data = json.load(f)

        rows = []
        for key, entry in raw_data.items():
            word = entry.get('homonym', '')
            sentence = entry.get('sentence', '')

            # Replace the target word with [MASK]
            masked_sentence = sentence.replace(word, tokenizer.mask_token)
            if masked_sentence == sentence: # Try capitalized version if no match
                masked_sentence = sentence.replace(word.capitalize(), tokenizer.mask_token)

            prompt = (
                f"Narrative: {entry.get('precontext', '')} {masked_sentence} {entry.get('ending', '')} "
                f"Question: How plausible is this meaning for the missing word? "
                f"Meaning: {entry.get('judged_meaning', '')}"
            )
            if split in ['train', 'dev']:
              rows.append({
                  "text_input": prompt,
                  "label": float(entry.get('average', 0.0))
              })
            else:
              rows.append({
                  "text_input": prompt
              })

        dataset[split] = pd.DataFrame(rows)
        print(f"Successfully loaded {split}: {len(dataset[split])} rows.")

    return dataset

data_dict_masked = load_masked_ambistory("data")
train_df = data_dict_masked['train']
dev_df = data_dict_masked['dev']
test_df = data_dict_masked['test']

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Successfully loaded train: 2280 rows.
Successfully loaded dev: 588 rows.
Successfully loaded test: 930 rows.


In [12]:
sample_text = train_df['text_input'].iloc[0]
tokens = tokenizer.tokenize(sample_text)
print(f"Token length: {len(tokens)}")
# check if the input prompt is not too large (i.e. >256 characters)

Token length: 89


In [9]:
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.flatten()

    # mse = mean_squared_error(labels, predictions)
    try:
        spearman_corr = spearmanr(predictions, labels)[0]
    except Exception:
        spearman_corr = 0.0

    # Handle cases where spearmanr returns NaN (due to constant predictions)
    if np.isnan(spearman_corr):
        spearman_corr = 0.0

    return {
        # "mse": mse,
        "spearman": spearman_corr
    }

In [13]:
# MODEL_NAME = "microsoft/deberta-v3-base"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text_input"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
dev_dataset = Dataset.from_pandas(dev_df).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

training_args = TrainingArguments(
    output_dir="./ambistory_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    warmup_ratio=0.1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print("Generating validation predictions...")
predictions = trainer.predict(dev_dataset)
preds = predictions.predictions.flatten()
labels = predictions.label_ids.flatten()

preds = np.clip(preds, 1.0, 5.0)
results_df = pd.DataFrame({
    'predicted_score': preds
})

spearman_rho, _ = spearmanr(preds, labels)
mae = mean_absolute_error(labels, preds)
mse = mean_squared_error(labels, preds)
rmse = np.sqrt(mse)

print("\n--- Model Quality Metrics On dev Dataset---")
print(f"Spearman Correlation (Rho): {spearman_rho:.4f}")
print(f"Mean Absolute Error (MAE):  {mae:.4f}")
print(f"Root Mean Squared Error:    {rmse:.4f}")

print("\nSample of predictions:")
display(results_df.head())

# output_filename = "model_predictions_on_dev.csv"
# results_df.to_csv(output_filename, index=False)
# print(f"\nPredictions saved to {output_filename}")

# files.download(output_filename)
# print("File download initiated.")

Generating validation predictions...



--- Model Quality Metrics On dev Dataset---
Spearman Correlation (Rho): 0.4026
Mean Absolute Error (MAE):  0.9969
Root Mean Squared Error:    1.2497

Sample of predictions:


,predicted_score
0,3.449219
1,3.750000
2,2.314453
3,4.976562
4,2.691406


In [ ]:
%cd /content

/content


In [ ]:
print("Generating test predictions...")
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.flatten()

preds = np.clip(preds, 1.0, 5.0)
preds = np.round(preds).astype(int) # Round to nearest integer

# # Get the IDs from the original test_df
# test_ids = test_df['id'].tolist()

# Create a list of dictionaries for JSON Lines output
json_results = []
for i, pred in enumerate(preds):
    json_results.append({"id": i, "prediction": int(pred)})

print("\nSample of predictions (JSON Lines format):")
for i in range(min(5, len(json_results))):
    print(json.dumps(json_results[i]))

output_filename = "predictions.json"
with open(output_filename, 'w') as f:
    for entry in json_results:
        f.write(json.dumps(entry) + '\n')
print(f"\nPredictions saved to {output_filename}")

files.download(output_filename)
print("File download initiated.")

Generating test predictions...



Sample of predictions (JSON Lines format):
{"id": 0, "prediction": 5}
{"id": 1, "prediction": 4}
{"id": 2, "prediction": 4}
{"id": 3, "prediction": 4}
{"id": 4, "prediction": 4}

Predictions saved to model_predictions_on_test.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File download initiated.


In [ ]:
def test_plausibility(context, sentence, ending, meaning):
    model.eval()
    # simple prompt
    text = f"{context} {sentence} {ending} [SEP] {meaning}"
    # instructional prompt
    # text = (
    #       f"Narrative: {entry.get('precontext', '')} {entry.get('sentence', '')} {entry.get('ending', '')} "
    #       f"Question: On a scale of 1 to 5, how plausible is the following meaning for the word '{entry.get('homonym', '')}'? "
    #       f"Meaning: {entry.get('judged_meaning', '')}"
    #   )

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to("cuda")

    with torch.no_grad():
        output = model(**inputs)
        score = output.logits.item()
    final_score = max(1.0, min(5.0, score))
    print(f"\nNarrative: {context} {sentence} {ending}")
    print(f"Tested Meaning: {meaning}")
    print(f"Model Plausibility Rating: {final_score:.2f} / 5.0")


test_plausibility(
    context="The old machine hummed in the workshop. Clara examined the dials.",
    sentence="The potential couldn't be measured.",
    ending="She collected a battery reader and looked on earnestly.",
    meaning="the difference in electrical charge between two points"
)

In [ ]:
local_path = "../my_final_model"

trainer.save_model(local_path)
tokenizer.save_pretrained(local_path)

print(f"Model saved locally to the folder: {local_path}")

Model saved locally to the folder: ../my_final_model


In [ ]:
shutil.make_archive("semeval_model_archive", 'zip', local_path)
files.download("semeval_model_archive.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
path = "/content/drive/MyDrive/SemEval_Best_Model"
loaded_model = AutoModelForSequenceClassification.from_pretrained(path)
loaded_tokenizer = AutoTokenizer.from_pretrained(path)

print("Model reloaded and ready for inference!")

### Ensemble Model Explanation

1.  **DeBERTa Encoder (PlausibilityRegressor)**: A fine-tuned `microsoft/deberta-v3-base` model is used as an encoder. Its primary role is to process the narrative context and the meaning, generating a regression score for plausibility. This model is trained specifically on the task's dataset to capture direct relationships between input text and plausibility ratings.

2.  **Mistral-7B Large Language Model (LLM)**: A `mistralai/Mistral-7B-v0.1` model, loaded in 4-bit quantization, acts as the second component. It is prompted with the narrative and meaning, then asked to provide a plausibility score. The LLM's strength lies in its broad understanding of language and common sense reasoning, which can complement the fine-tuned encoder.


**Blending Strategy**: The predictions from both models are combined using a weighted average. In the current implementation, the DeBERTa encoder's prediction is given a weight of 0.7, and the Mistral-7B LLM's prediction receives a weight of 0.3. This blending aims to achieve a more robust and accurate final plausibility score by integrating both specialized knowledge (from DeBERTa) and general world knowledge (from Mistral-7B).

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (AutoModel, AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig)
from tqdm.auto import tqdm

# --- CONFIGURATION ---
# Using T4 x2? Model 1 on GPU 0, Model 2 on GPU 1.
DEVICE_ENC = "cuda:0"
DEVICE_LLM = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

# --- DATASET LOADER ---
class SemEvalDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Use the column name you defined in load_masked_ambistory
        text = row['text_input']

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=256,
            padding='max_length',
            return_tensors="pt"
        )

        res = {
            'ids': enc['input_ids'].squeeze(),
            'mask': enc['attention_mask'].squeeze(),
        }

        # Add labels only if they exist (for train/dev)
        if not self.is_test:
            res['label'] = torch.tensor(row['labels'], dtype=torch.float)

        return res

# --- ENCODER MODEL (DeBERTa) ---
class PlausibilityRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, ids, mask):
        out = self.backbone(ids, attention_mask=mask)
        # We take the [CLS] token and pass it through the regression head
        return self.head(out.last_hidden_state[:, 0, :])

# --- ENSEMBLE ENGINE ---
def run_ensemble(train_df):
    print("Step 1: Initializing DeBERTa Encoder...")
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
    model = PlausibilityRegressor("microsoft/deberta-v3-base").to(DEVICE_ENC)

    print("Step 2: Loading Mistral-7B in 4-bit (This takes ~2 mins)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    # device_map="auto" or specific map to handle memory
    llm = AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-7B-v0.1",
        quantization_config=bnb_config,
        device_map={"": DEVICE_LLM}
    )

    # Step 3: Training Loop
    loader = DataLoader(SemEvalDataset(train_df, tokenizer), batch_size=4, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.MSELoss()

    model.train()
    print("Step 3: Training Encoder on Narrative Context...")
    for epoch in range(1): # Set to 3+ for real training
        for batch in tqdm(loader):
            optimizer.zero_grad()
            preds = model(batch['ids'].to(DEVICE_ENC), batch['mask'].to(DEVICE_ENC))
            loss = criterion(preds, batch['label'].to(DEVICE_ENC).view(-1, 1))
            loss.backward()
            optimizer.step()

    return model, llm, tokenizer


trained_encoder, loaded_llm, common_tokenizer = run_ensemble(train_df)

print("\n" + "-"*30)
print("SUCCESS: ENSEMBLE LOADED")
print(f"Encoder VRAM: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")
print(f"LLM VRAM: {torch.cuda.memory_allocated(1)/1024**2 if torch.cuda.device_count() > 1 else 0:.2f} MB")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import re
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

# 1. SETTINGS
DEVICE_ENC = "cuda:0"
DEVICE_LLM = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

# 2. SEPARATE TOKENIZERS (The most important part)
print("Loading Tokenizers...")
enc_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
llm_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
llm_tokenizer.pad_token = llm_tokenizer.eos_token

# 3. LOAD MODELS
print("Loading DeBERTa...")
# Note: Ensure your trained_encoder is loaded here.
# If you lost it during restart, you must re-run your training cell first.
encoder = trained_encoder.to(DEVICE_ENC).eval()

print("Loading Mistral 4-bit...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map={"": DEVICE_LLM}
)

# 4. ROBUST PREDICTION FUNCTION
def get_safe_predictions(df):
    final_results = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text = row['text_input']

        # --- ENCODER INFERENCE ---
        # Explicitly use the ENCODER tokenizer
        enc_inputs = enc_tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE_ENC)
        with torch.inference_mode():
            # Pass only the keys DeBERTa expects
            d_out = encoder(enc_inputs['input_ids'], enc_inputs['attention_mask'])
            deberta_score = d_out.item()

        # --- LLM INFERENCE ---
        # Explicitly use the LLM tokenizer
        llm_inputs = llm_tokenizer(f"{text}\nScore (1-5):", return_tensors="pt").to(DEVICE_LLM)
        # Remove token_type_ids which cause the crash in Mistral
        llm_inputs.pop("token_type_ids", None)

        with torch.inference_mode():
            outputs = llm.generate(**llm_inputs, max_new_tokens=2, pad_token_id=llm_tokenizer.eos_token_id)
            response = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract number 1-5
        match = re.search(r'[1-5]', response.split("Score (1-5):")[-1])
        llm_score = float(match.group()) if match else 3.0

        # --- BLEND ---
        combined = (0.7 * deberta_score) + (0.3 * llm_score)
        final_results.append(np.clip(combined, 1.0, 5.0))

        # Periodic VRAM clear to prevent "Assert" errors
        if idx % 50 == 0:
            torch.cuda.empty_cache()

    return final_results

# 5. SEE THE DATA
test_preds = get_safe_predictions(test_df)
test_df['final_prediction'] = test_preds

print("\n--- TEST PREDICTIONS ---")
print(test_df[['text_input', 'final_prediction']].head(10))

# Save
test_df[['final_prediction']].to_csv('submission.csv', index=False)

In [ ]:
import pandas as pd

# 1. Create the DataFrame
# Ensure 'id' matches the original test set keys
submission_df = pd.DataFrame({
    "id": test_df.index.astype(str),
    "prediction": test_preds
})

# 2. Convert floats to rounded integers
# .round() follows standard rounding rules (.5 goes up)
submission_df["prediction"] = submission_df["prediction"].round().astype(int)

# 3. Export to JSONL format
submission_df.to_json(
    "submission.jsonl",
    orient="records",
    lines=True
)

print("File saved as submission.jsonl!")
# Verify the format
!head -n 5 submission.jsonl

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'true_score': labels,
    'predicted_score': preds,
    'abs_error': np.abs(labels - preds)
})

# --- CALCULATION ---
spearman_rho, _ = spearmanr(preds, labels)
mae = mean_absolute_error(labels, preds)
mse = mean_squared_error(labels, preds)
rmse = np.sqrt(mse)

# SemEval Primary Metric: Accuracy within Standard Deviation (SD)
# Most tasks define this as |pred - true| <= 1.0 (or the specific SD provided in data)
within_sd = np.mean(results_df['abs_error'] <= 1.0) * 100

print("\n--- Model Quality Metrics On Dev Dataset ---")
print(f"Spearman Correlation (Rho): {spearman_rho:.4f} (Ranking performance)")
print(f"Accuracy within 1.0 SD:    {within_sd:.2f}% (Main Leaderboard Metric)")
print(f"Mean Absolute Error (MAE):  {mae:.4f}")
print(f"Root Mean Squared Error:    {rmse:.4f}")

print("\nSample of predictions:")
display(results_df.head(10))